# Technischer GPU-Check — keine Modellgewichte nötig

**GPU auswählen → Alle ausführen → Ergebnis-ZIP herunterladen.** A100 oder L4 verwenden. T4 wird klar abgelehnt; es gibt keine Warteschleife.

Das Notebook prüft BF16-Hooks auf einem **zufällig initialisierten** nativen Qwen3.5-MoE mit zwei Layern und acht Experten. Hidden-, Head- und MLP-Dimensionen entsprechen dem vorgesehenen Modell; Layer-/Expertenzahl und Routing sind reduziert. Erwarteter GPU-Speicher unter etwa 3 GiB, ohne Garantie. Es lädt weder veröffentlichte Gewichte noch Drive-/Atlas-/H286-Daten.

Geprüft werden Sham-Identität, Query-Puls **nach** q_norm bei unverändertem Projektions-Gate, Shared-Aktivierungen, routed Schattenberechnung gegen den tatsächlich beobachteten nativen Expertenbatch sowie RNG und Hook-Cleanup. Toleranzen sind technische Prüfschwellen, keine wissenschaftliche Kalibrierung. Source ist auf `4ffee35204a6e0c0626ecfa4b2a2e1371a47bfd4`, Transformers auf `5.11.0` festgelegt. Die vorhandene Colab-Torch/CUDA-Version wird protokolliert und tatsächlich getestet, ohne sie still zu ersetzen. Das bestätigt keine Atlas-Koordinatenabbildung und keinen vollständigen Checkpointlauf.

Auch bei einem Fehler wird ein Diagnose-ZIP mit Status, Messwerten und Log vorbereitet. Nach dem Test GPU-Laufzeit bei Bedarf selbst beenden.


In [ ]:
# GPU auswählen → Alle ausführen → Ergebnis-ZIP wird heruntergeladen.
import datetime, json, os, pathlib, subprocess, sys, tempfile, traceback, uuid, zipfile
SOURCE_COMMIT = '4ffee35204a6e0c0626ecfa4b2a2e1371a47bfd4'
CPU_SMOKE = False  # Nur für lokale Entwicklertests; qualifiziert keine GPU.
OUT = pathlib.Path('/content/atlas_gpu_check') / (datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8])
OUT.mkdir(parents=True, exist_ok=True)

def emergency_bundle(exc):
    (OUT / 'status.json').write_text(json.dumps({'state': 'failed', 'stage': 'setup', 'gpu_qualified': False, 'error_type': type(exc).__name__, 'error': str(exc)}, indent=2))
    (OUT / 'traceback.txt').write_text(traceback.format_exc())
    if not (OUT / 'metrics.json').exists():
        (OUT / 'metrics.json').write_text(json.dumps({'purpose': 'technical_random_model_check', 'checks': {}, 'pretrained_weights_loaded': False}, indent=2))
    with (OUT / 'setup.log').open('a') as setup_log:
        setup_log.write(traceback.format_exc())
    target = OUT / 'atlas_gpu_check.zip'
    with zipfile.ZipFile(target, 'w', zipfile.ZIP_DEFLATED) as z:
        for item in OUT.iterdir():
            if item.is_file() and item != target:
                z.write(item, item.name)
    print('FEHLER. Diagnose:', target)
    try:
        from google.colab import files
        files.download(str(target))
    except Exception as download_error:
        print('Download bei Bedarf im Colab-Dateibrowser:', target, type(download_error).__name__)

try:
    import torch
    print('Torch:', torch.__version__)
    if not CPU_SMOKE:
        if not torch.cuda.is_available():
            raise RuntimeError('Keine GPU. Colab: Laufzeit → Laufzeittyp ändern → A100 oder L4. Kein Warten.')
        capability = torch.cuda.get_device_capability()
        print('GPU:', torch.cuda.get_device_name(), 'Capability:', capability)
        if capability[0] < 8 or not torch.cuda.is_bf16_supported():
            raise RuntimeError('A100/L4 oder andere native BF16-GPU erforderlich. T4 wird nicht unterstützt.')
    # Torch bleibt die von Colab bereitgestellte CUDA-Installation; kein stiller Upgrade.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'transformers==5.11.0'], check=True)
    REPO = pathlib.Path(tempfile.mkdtemp(prefix='atlas_gpu_source_', dir='/content'))
    def git(*args):
        return subprocess.run(['git', '-C', str(REPO), *args], check=True, capture_output=True, text=True).stdout.strip()
    git('init', '--quiet')
    git('remote', 'add', 'origin', 'https://github.com/Erikiss/Interface-Neuron-Atlas-by-GoodFire-AI.git')
    git('fetch', '--quiet', '--depth', '1', 'origin', SOURCE_COMMIT)
    git('checkout', '--quiet', '--detach', 'FETCH_HEAD')
    if git('rev-parse', 'HEAD') != SOURCE_COMMIT:
        raise RuntimeError('Source-Commit stimmt nicht überein.')
    (OUT / 'source.json').write_text(json.dumps({'commit': SOURCE_COMMIT, 'transformers': '5.11.0', 'weights_loaded': False}, indent=2))
    print('Quellcode geprüft:', SOURCE_COMMIT)
except BaseException as exc:
    emergency_bundle(exc)
    raise RuntimeError('Setup abgebrochen; Diagnose-ZIP wurde vorbereitet.') from None


In [ ]:
# Eingebetteter Prüfcode; benötigt keine unveröffentlichten Repository-Dateien.
HELPER_SOURCE = '"""Embedded technical adapter check. Random native models only; never loads weights."""\nfrom __future__ import annotations\n\nimport argparse\nimport contextlib\nimport hashlib\nimport json\nimport platform\nimport time\nimport traceback\nimport zipfile\nfrom pathlib import Path\n\n\ndef bundle(output: Path) -> Path:\n    target = output / \'atlas_gpu_check.zip\'\n    with zipfile.ZipFile(target, \'w\', zipfile.ZIP_DEFLATED) as archive:\n        for item in sorted(output.iterdir()):\n            if item.is_file() and item != target:\n                archive.write(item, item.name)\n    return target\n\n\ndef run_check(outputdir, *, use_gpu=True, production_shapes=True):\n    """CPU smoke mode is explicit; its success never qualifies a GPU runtime."""\n    output = Path(outputdir)\n    output.mkdir(parents=True, exist_ok=True)\n    metrics = {\'purpose\': \'technical_random_model_check\', \'atlas_measurements\': False,\n               \'pretrained_weights_loaded\': False, \'atlas_mapping_verified\': False,\n               \'production_shapes\': production_shapes, \'checks\': {}}\n    cleanup_handles = []\n    original_linear = None\n    started = time.monotonic()\n    status = {\'state\': \'running\', \'stage\': \'imports\', \'gpu_qualified\': False}\n\n    def save():\n        (output / \'status.json\').write_text(json.dumps(status, indent=2) + \'\\n\')\n        (output / \'metrics.json\').write_text(json.dumps(metrics, indent=2, allow_nan=False) + \'\\n\')\n\n    def check(name, passed, detail=None):\n        metrics[\'checks\'][name] = {\'passed\': bool(passed), \'detail\': detail}\n        print((\'PASS \' if passed else \'FAIL \') + name, flush=True)\n        if not passed:\n            raise AssertionError(name)\n\n    save()\n    try:\n        import torch\n        import transformers\n        from transformers.models.qwen3_5_moe.configuration_qwen3_5_moe import Qwen3_5MoeTextConfig\n        from transformers.models.qwen3_5_moe.modeling_qwen3_5_moe import Qwen3_5MoeForCausalLM\n\n        import neuron_atlas.runtime_adapter as adapter\n        from neuron_atlas.runtime_adapter import FullAttentionPulse, MoeActivationCapture\n\n        original_linear = torch.nn.functional.linear\n        check(\'transformers_pin\', transformers.__version__ == \'5.11.0\')\n        metrics.update(python=platform.python_version(), torch=torch.__version__,\n                       transformers=transformers.__version__, adapter_sha256=hashlib.sha256(\n                           Path(adapter.__file__).read_bytes()).hexdigest())\n        if use_gpu:\n            if not torch.cuda.is_available():\n                raise RuntimeError(\'Keine GPU: Colab-Laufzeit auf A100 oder L4 stellen; kein Warten.\')\n            capability = torch.cuda.get_device_capability()\n            if capability[0] < 8 or not torch.cuda.is_bf16_supported():\n                raise RuntimeError(\'Native BF16-GPU erforderlich. T4/Float16 wird hier nicht qualifiziert.\')\n            device, dtype = \'cuda\', torch.bfloat16\n            torch.backends.cuda.matmul.allow_tf32 = False\n            torch.backends.cudnn.allow_tf32 = False\n            metrics.update(gpu=torch.cuda.get_device_name(), compute_capability=list(capability),\n                           native_bf16_hardware=True)\n            torch.cuda.reset_peak_memory_stats()\n        else:\n            device, dtype = \'cpu\', torch.float32\n            torch.set_num_threads(2)\n        metrics.update(device=device, dtype=str(dtype))\n        config_values = {\n            \'vocab_size\': 32, \'hidden_size\': 2048, \'num_attention_heads\': 16, \'num_key_value_heads\': 2,\n            \'head_dim\': 256, \'moe_intermediate_size\': 512, \'shared_expert_intermediate_size\': 512,\n            \'num_experts\': 8, \'num_experts_per_tok\': 2, \'num_hidden_layers\': 2,\n            \'layer_types\': [\'linear_attention\', \'full_attention\'],\n            \'linear_key_head_dim\': 128, \'linear_value_head_dim\': 128,\n            \'linear_num_key_heads\': 16, \'linear_num_value_heads\': 32,\n            \'rope_parameters\': {\'rope_type\': \'default\', \'rope_theta\': 10000000.,\n                             \'partial_rotary_factor\': .25, \'mrope_section\': [11, 11, 10]},\n        }\n        if not production_shapes:\n            config_values.update(hidden_size=16, num_attention_heads=2, num_key_value_heads=1,\n                                 head_dim=8, moe_intermediate_size=4, shared_expert_intermediate_size=4,\n                                 num_experts=3, linear_key_head_dim=4, linear_value_head_dim=4,\n                                 linear_num_key_heads=2, linear_num_value_heads=2,\n                                 rope_parameters={\'rope_type\': \'default\', \'rope_theta\': 10000.,\n                                                  \'partial_rotary_factor\': .5, \'mrope_section\': [1, 1, 0]})\n        metrics[\'random_model_config\'] = config_values\n        metrics[\'seed\'] = 412\n        status[\'stage\'] = \'random_model\'\n        save()\n        torch.manual_seed(412)\n        config = Qwen3_5MoeTextConfig(**config_values)\n        config._attn_implementation = \'eager\'\n        config._experts_implementation = \'eager\'\n        model = Qwen3_5MoeForCausalLM(config).eval().to(device=device, dtype=dtype)\n        inputs = torch.tensor([[1, 2, 3, 4, 5, 6, 7, 8], [8, 7, 6, 5, 4, 3, 2, 1]], device=device)\n        positions = [2, 6]\n        mlp = model.model.layers[1].mlp\n        attention = model.model.layers[1].self_attn\n        coords = [{\'layer\': 1, \'expert\': expert, \'neuron\': neuron, \'kind\': \'routed\'}\n                  for expert in range(config.num_experts) for neuron in [0, config.moe_intermediate_size - 1]]\n        coords += [{\'layer\': 1, \'expert\': None, \'neuron\': n, \'kind\': \'shared\'}\n                   for n in [0, config.shared_expert_intermediate_size - 1]]\n\n        def forward():\n            return model(inputs, use_cache=False).logits.detach()\n\n        def count_hooks():\n            return sum(len(module._forward_hooks) + len(module._forward_pre_hooks) for module in model.modules())\n\n        @contextlib.contextmanager\n        def observe_native():\n            # This marker is installed AFTER the adapter\'s prehook: its shadow\n            # GEMMs run with active=False. Only the actual native forward is observed.\n            state = {\'active\': False, \'expert\': None}\n            seen, handles = {}, []\n            original = torch.nn.functional.linear\n            gate_ptrs = {mlp.experts.gate_up_proj[e].data_ptr(): e for e in range(config.num_experts)}\n            down_ptrs = {mlp.experts.down_proj[e].data_ptr(): e for e in range(config.num_experts)}\n\n            def begin(_module, args):\n                state[\'active\'] = True\n                hidden, indices, weights = args\n                seen[\'routing\'] = (indices.detach().cpu().clone(), weights.detach().float().cpu().clone())\n\n            def end(_module, _args, _output):\n                state[\'active\'] = False\n\n            def linear(x, weight, *args, **kwargs):\n                if state[\'active\'] and weight.data_ptr() in down_ptrs:\n                    expert = down_ptrs[weight.data_ptr()]\n                    seen[expert, \'raw_z\'] = x.detach().float().cpu().clone()\n                result = original(x, weight, *args, **kwargs)\n                if state[\'active\'] and weight.data_ptr() in gate_ptrs:\n                    expert = gate_ptrs[weight.data_ptr()]\n                    state[\'expert\'] = expert\n                    gate, up = result.chunk(2, dim=-1)\n                    seen[expert, \'gate_pre\'] = gate.detach().float().cpu().clone()\n                    seen[expert, \'up\'] = up.detach().float().cpu().clone()\n                return result\n\n            def activated(_module, _args, value):\n                if state[\'active\']:\n                    seen[state[\'expert\'], \'gate_post\'] = value.detach().float().cpu().clone()\n\n            def shared_down(_module, args):\n                seen[\'shared\'] = args[0].detach().float().cpu().clone()\n\n            try:\n                handles.append(mlp.experts.register_forward_pre_hook(begin))\n                handles.append(mlp.experts.register_forward_hook(end, always_call=True))\n                handles.append(mlp.experts.act_fn.register_forward_hook(activated))\n                handles.append(mlp.shared_expert.down_proj.register_forward_pre_hook(shared_down))\n                torch.nn.functional.linear = linear\n                yield seen\n            finally:\n                state[\'active\'] = False\n                torch.nn.functional.linear = original\n                for handle in reversed(handles):\n                    handle.remove()\n\n        cpu_rng = torch.random.get_rng_state().clone()\n        cuda_rng = torch.cuda.get_rng_state().clone() if use_gpu else None\n        initial_hooks = count_hooks()\n        baseline_q, baseline_gate = [], []\n        q_hook = attention.q_norm.register_forward_hook(lambda _m, _a, out: baseline_q.append(out.detach().clone()))\n        gate_hook = attention.q_proj.register_forward_hook(lambda _m, _a, out: baseline_gate.append(out.detach().clone()))\n        cleanup_handles.extend([q_hook, gate_hook])\n        status[\'stage\'] = \'native_sham_and_capture\'\n        save()\n        with torch.inference_mode():\n            baseline = forward()\n            with (MoeActivationCapture(model, coords, positions) as capture,\n                  FullAttentionPulse(model, 1, positions, beta=1) as sham, observe_native() as native):\n                sham_output = forward()\n        q_hook.remove()\n        gate_hook.remove()\n        check(\'sham_output_bitwise_equal\', torch.equal(baseline, sham_output))\n        check(\'sham_once\', sham.fired_count == sham.completed_count == capture.fired_count == 1)\n        routed_pairs = {key: ([], []) for key in (\'raw_z\', \'gate_pre\', \'gate_post\', \'up\')}\n        shared_equal, null_count = True, 0\n        indices, weights = native[\'routing\']\n        for row in capture.records:\n            token = row[\'batch_index\'] * inputs.shape[1] + row[\'position\']\n            if row[\'expert\'] is None:\n                shared_equal &= row[\'raw_z\'] == float(native[\'shared\'][token, row[\'neuron\']])\n                continue\n            expert = row[\'expert\']\n            native_ranks, native_tokens = torch.where((indices == expert).T)\n            matched = (native_tokens == token).nonzero()\n            if not matched.numel():\n                null_count += 1\n                check_value = row[\'routed\'] is False and row[\'raw_z\'] is None\n                if not check_value:\n                    raise AssertionError(\'Not-routed record must be null, never a zero activation\')\n                continue\n            native_row = int(matched[0, 0])\n            rank = int(native_ranks[native_row])\n            if not row[\'routed\'] or row[\'routing_rank\'] != rank or row[\'routing_weight\'] != float(weights[token, rank]):\n                raise AssertionError(\'Native routing identity/rank/weight mismatch\')\n            for key, (shadow, actual) in routed_pairs.items():\n                shadow.append(row[key])\n                actual.append(float(native[expert, key][native_row, row[\'neuron\']]))\n        check(\'shared_native_down_input_exact\', shared_equal)\n        check(\'not_routed_is_null\', null_count > 0, {\'records\': null_count})\n        check(\'at_least_one_routed_comparison\', len(routed_pairs[\'raw_z\'][0]) > 0)\n        metrics[\'routed_comparison\'] = {}\n        rtol, atol = ((.03, .02) if use_gpu else (1e-4, 1e-5))\n        for key, (shadow, actual) in routed_pairs.items():\n            shadow, actual = torch.tensor(shadow), torch.tensor(actual)\n            difference = (shadow - actual).abs()\n            detail = {\'n\': len(shadow), \'max_abs\': float(difference.max()),\n                      \'rms_error\': float(difference.square().mean().sqrt()),\n                      \'native_reference_rms\': float(actual.square().mean().sqrt()),\n                      \'native_reference_max_abs\': float(actual.abs().max()),\n                      \'max_rel_floor_1e-6\': float((difference / actual.abs().clamp_min(1e-6)).max()),\n                      \'rtol\': rtol, \'atol\': atol,\n                      \'reference\': \'actual_native_expert_forward_full_routed_batch\',\n                      \'shadow\': \'adapter_selected_token_gemm\',\n                      \'threshold_status\': \'engineering_smoke_threshold_not_scientific_calibration\'}\n            metrics[\'routed_comparison\'][key] = detail\n            check(\'routed_\' + key + \'_parity\', torch.allclose(shadow, actual, rtol=rtol, atol=atol), detail)\n        changed_q, changed_gate = [], []\n        status[\'stage\'] = \'post_norm_pulse\'\n        save()\n        with torch.inference_mode(), FullAttentionPulse(model, 1, positions, beta=.9) as pulse:\n            handles = [attention.q_norm.register_forward_hook(lambda _m, _a, out: changed_q.append(out.detach().clone())),\n                       attention.q_proj.register_forward_hook(lambda _m, _a, out: changed_gate.append(out.detach().clone()))]\n            cleanup_handles.extend(handles)\n            try:\n                pulsed_output = forward()\n            finally:\n                for handle in handles:\n                    handle.remove()\n        expected = baseline_q[0].clone()\n        expected[:, positions] *= .9\n        check(\'pulse_exactly_after_q_norm\', torch.equal(changed_q[0], expected))\n        check(\'q_proj_including_output_gate_unchanged\', torch.equal(changed_gate[0], baseline_gate[0]))\n        check(\'pulse_finite_and_effective\', bool(torch.isfinite(pulsed_output).all()) and not torch.equal(pulsed_output, baseline))\n        check(\'causal_earlier_tokens_unchanged\', torch.equal(pulsed_output[:, :min(positions)], baseline[:, :min(positions)]))\n        with torch.inference_mode(), FullAttentionPulse(model, 1, positions, beta=.9, mode=\'random\', seed=9) as random:\n            forward()\n        metrics[\'random_delta_norms\'] = random.norm_records\n        check(\'random_norm_match\', all(abs(row[\'applied_delta_norm\'] - row[\'requested_delta_norm\'])\n                                     <= (0.04 * row[\'requested_delta_norm\'] + 1e-6)\n                                     for row in random.norm_records))\n        check(\'cpu_rng_unchanged\', torch.equal(cpu_rng, torch.random.get_rng_state()))\n        if use_gpu:\n            check(\'cuda_rng_unchanged\', torch.equal(cuda_rng, torch.cuda.get_rng_state()))\n            metrics[\'peak_gpu_allocated_bytes\'] = torch.cuda.max_memory_allocated()\n            metrics[\'peak_gpu_reserved_bytes\'] = torch.cuda.max_memory_reserved()\n        check(\'all_hooks_cleaned\', count_hooks() == initial_hooks)\n        check(\'functional_linear_spy_restored\', torch.nn.functional.linear is original_linear)\n        metrics[\'pulse_delta_norms\'] = pulse.norm_records\n        status.update(state=\'complete\', stage=\'complete\', gpu_qualified=bool(use_gpu),\n                      qualification_scope=\'random_reduced_native_model_only\')\n    except BaseException as exc:\n        status.update(state=\'failed\', stage=status[\'stage\'], error_type=type(exc).__name__, error=str(exc))\n        (output / \'traceback.txt\').write_text(traceback.format_exc())\n        print(\'HARD FAILURE: \' + str(exc), flush=True)\n    finally:\n        for handle in reversed(cleanup_handles):\n            handle.remove()\n        if original_linear is not None:\n            torch.nn.functional.linear = original_linear\n        metrics[\'elapsed_seconds\'] = time.monotonic() - started\n        save()\n        bundle(output)\n    return status\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--outputdir\', required=True)\n    parser.add_argument(\'--cpu-smoke\', action=\'store_true\')\n    args = parser.parse_args()\n    result = run_check(args.outputdir, use_gpu=not args.cpu_smoke, production_shapes=not args.cpu_smoke)\n    print(json.dumps(result), flush=True)\n    raise SystemExit(0 if result[\'state\'] == \'complete\' else 1)\n'
HELPER_PATH = OUT / "technical_check.py"
HELPER_PATH.write_text(HELPER_SOURCE)


In [ ]:
# Frischer Python-Prozess nach Paketinstallation; keine geerbten HF-Imports.
command = [sys.executable, str(HELPER_PATH), '--outputdir', str(OUT)]
if CPU_SMOKE:
    command.append('--cpu-smoke')
env = dict(os.environ)
env['PYTHONPATH'] = str(REPO / 'src')
env['TOKENIZERS_PARALLELISM'] = 'false'
env['HF_HUB_OFFLINE'] = '1'
env['TRANSFORMERS_OFFLINE'] = '1'
print('Technische Prüfung startet; Zufallsmodell, keine Atlas-Messung.')
try:
    with (OUT / 'run.log').open('w') as log:
        process = subprocess.Popen(command, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                log.write(line)
                log.flush()
                print(line.rstrip(), flush=True)
            RETURN_CODE = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
except BaseException as exc:
    emergency_bundle(exc)
    raise RuntimeError('Prüfung unterbrochen; Diagnose-ZIP wurde vorbereitet.') from None


In [ ]:
# Auch bei einem technischen FAIL werden Protokoll und Messwerte heruntergeladen.
ZIP = OUT / 'atlas_gpu_check.zip'
try:
    status = json.loads((OUT / 'status.json').read_text())
except Exception:
    status = {'state': 'failed', 'stage': 'child_process', 'gpu_qualified': False}
status['return_code'] = RETURN_CODE
if RETURN_CODE != 0 or status.get('state') != 'complete':
    status['child_state'] = status.get('state', 'missing')
    status['state'] = 'failed'
    status['gpu_qualified'] = False
    status.setdefault('stage', 'child_process')
(OUT / 'status.json').write_text(json.dumps(status, indent=2))
# run.log enthält auch die letzte Kindprozess-Ausgabe; ZIP deshalb neu abschließen.
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    for item in sorted(OUT.iterdir()):
        if item.is_file() and item != ZIP:
            z.write(item, item.name)
print('Status:', status['state'], '| GPU-Prüfung:', status.get('gpu_qualified', False))
print('Ergebnis:', ZIP)
try:
    from google.colab import files
    files.download(str(ZIP))
except Exception as exc:
    print('ZIP im Colab-Dateibrowser herunterladen:', ZIP, type(exc).__name__)
if RETURN_CODE or status['state'] != 'complete':
    raise RuntimeError('Technische Prüfung fehlgeschlagen. Das ZIP enthält den tatsächlichen Fehler.')
print('Technischer Zufallsmodell-Test bestanden; keine Ergebnisse auf echten Atlas-Gewichten.')
